modificar las columnas del csv original porque vienen sucias y spark no puede leerlo

In [4]:
import os
import glob

# ==============================================================================
# CONFIGURACIÓN
# ==============================================================================
base_dir = os.path.dirname(os.getcwd())
bronze_dir = os.path.join(base_dir, "data", "bronze")
silver_dir = os.path.join(base_dir, "data", "silver")
os.makedirs(silver_dir, exist_ok=True)

target_years = ["2018", "2019", "2020", "2021"]

# ==============================================================================
# FUNCIÓN: Limpiar línea de encabezado
# ==============================================================================
def clean_header_line(line):
    """
    1. Sustituye comas entre comillas por guiones
    2. Elimina TODOS los caracteres "
    3. Elimina el "# " inicial del encabezado
    """
    result = []
    in_quotes = False
    
    for char in line:
        if char == '"':
            in_quotes = not in_quotes
        elif char == ',' and in_quotes:
            result.append('-')
        else:
            result.append(char)
    
    cleaned = ''.join(result)
    # Quitar "# " inicial si existe
    if cleaned.startswith('# '):
        cleaned = cleaned[2:]
    elif cleaned.startswith('#'):
        cleaned = cleaned[1:]
    
    return cleaned

# ==============================================================================
# PROCESAR CADA ARCHIVO
# ==============================================================================
all_files = []
for year in target_years:
    pattern = os.path.join(bronze_dir, f"Kelmarsh_SCADA_{year}_*", f"Turbine_Data_Kelmarsh_1_*.csv")
    all_files.extend(glob.glob(pattern))

print(f"📁 Archivos a procesar: {len(all_files)}")

for file_path in all_files:
    filename = os.path.basename(file_path)
    silver_path = os.path.join(silver_dir, f"cleaned_{filename}")
    
    print(f"\n🔄 Procesando: {filename}")
    
    with open(file_path, 'r', encoding='utf-8') as infile, \
         open(silver_path, 'w', encoding='utf-8') as outfile:
        
        line_number = 0
        for line in infile:
            line_number += 1
            
            # ELIMINAR LÍNEAS 1-9 (comentarios de metadatos)
            if line_number < 10:
                continue
            
            if line_number == 10 and 'Date and time' in line:
                cleaned = clean_header_line(line)
                outfile.write(cleaned)
            else:
                outfile.write(line)
    
    print(f"   💾 Guardado en: {silver_path}")

print(f"\n✅ Todos los archivos procesados en: {silver_dir}")

📁 Archivos a procesar: 4

🔄 Procesando: Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv

🔄 Procesando: Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv

🔄 Procesando: Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv

🔄 Procesando: Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv
   💾 Guardado en: /home/aitor/Documentos/ai-driven-cross-generator-transfer-learning/data/silver/cleaned_Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv

✅ Todos los archivos procesados en: /home/aito

In [6]:
import os
import glob
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# ==============================================================================
# 1. SPARK SESSION
# ==============================================================================
spark = SparkSession.builder \
    .appName("Kelmarsh-EDA-Notebook") \
    .master("local[6]") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

# ==============================================================================
# 2. LECTURA DESDE CSV LIMPIOS
# ==============================================================================
base_dir = os.path.dirname(os.getcwd())
cleaned_dir = os.path.join(base_dir, "data", "silver")

target_years = ["2018", "2019", "2020", "2021"]
selected_paths = []
for year in target_years:
    pattern = os.path.join(cleaned_dir, f"cleaned_Turbine_Data_Kelmarsh_1_{year}*.csv")
    selected_paths.append(pattern)

all_files = []
for pattern in selected_paths:
    all_files.extend(glob.glob(pattern))

print(f"📁 Archivos encontrados: {len(all_files)}")
for f in all_files:
    print(f"   - {os.path.basename(f)}")

# ==============================================================================
# 3. CARGAR CON SPARK
# ==============================================================================
# Ahora el encabezado está en la línea 10 sin # inicial
# Las líneas 1-9 son comentarios (#), la 10 es el header válido
status_df = spark.read \
    .option("header", "True") \
    .option("inferSchema", "True") \
    .csv(selected_paths)

# ==============================================================================
# 4. RESULTADOS
# ==============================================================================

print("\nVISTA PREVIA DE LAS PRIMERAS 5 FILAS:")
status_df.show(5, truncate=False)

total_records = status_df.count()
print(f"\n🎯 Total de registros: {total_records}")
print(f"📊 NÚMERO TOTAL DE COLUMNAS: {len(status_df.columns)}")

📁 Archivos encontrados: 4
   - cleaned_Turbine_Data_Kelmarsh_1_2018-01-01_-_2019-01-01_228.csv
   - cleaned_Turbine_Data_Kelmarsh_1_2019-01-01_-_2020-01-01_228.csv
   - cleaned_Turbine_Data_Kelmarsh_1_2020-01-01_-_2021-01-01_228.csv
   - cleaned_Turbine_Data_Kelmarsh_1_2021-01-01_-_2022-01-01_228.csv



VISTA PREVIA DE LAS PRIMERAS 5 FILAS:
+-------------------+------------------+------------------------------------+-------------------------+-------------------------+--------------------+-------------------------+---------------------------------------------+----------------------------------+----------------------------------+-------------------------+---------------------------------------------+----------------------------------+----------------------------------+---------------------------------+------------------+--------------------+--------------------------------------+---------------------------+---------------------------+----------------------------------------+-----------------------------+-----------------------------+---------------------+--------------------------+--------------------------+-----------------------------+-------------------+---------------------------+-------------------+---------------------------+-----------------------------------+-------------------